In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
# 실습문제#4: 프롬프트 평가(prompt_evulate)의 종합예제. 채점 기준(solution_criteria)을 데이터셋에 포함시켜 Claude가 기준에 따라 코드를 자동 채점하게 만들기
# Function to generate a new dataset3
import json

def generate_dataset():
    # 각각 원본 '과제', '형식', '평가_기준'이 간결하게 기술된 데이터셋 생성 
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [12]:
# Generate the dataset3 and write it to 'dataset.json'

dataset3 = generate_dataset()
with open("dataset3.json", "w") as f:
    json.dump(dataset3, f, indent=2)

In [13]:
# 모델을 사용하여 테스트 케이스에 대한 클로드의 답변 을 평가하는 채점 로직
def grade_by_model(test_case, output):
    eval_prompt = f"""
당신은 숙련된 AWS 코드 리뷰어입니다. 아래 AI가 생성한 솔루션을 평가해주세요.

원본 과제:
<task>
{test_case["task"]}
</task>

평가 대상 솔루션:
<solution>
{output}
</solution>

솔루션을 평가하기 위해 이용해야 하는 평가기준:
<criteria>
{test_case["solution_criteria"]}
</criteria>

출력 형식
다음 필드를 포함한 구조화된 JSON 객체로 평가 결과를 작성하세요. 필드 순서는 아래와 같이 유지합니다:
- "strengths": 핵심 강점 1~3개를 담은 배열
- "weaknesses": 개선이 필요한 영역 1~3개를 담은 배열
- "reasoning": 전반적인 평가에 대한 간결한 설명
- "score": 1~10 사이의 숫자

JSON 형식으로만 응답하세요. 답변은 간결하고 직접적으로 작성합니다.

응답 예시 구조:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """
    messages = []
    add_user_message(messages, eval_prompt)
    # haiku 모델은 여전히 message_prefilling을 지원하므로 강의 예시대로 이를 사용함
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [14]:
# format 검사를 위한 코드 기반 채점 함수들 / 학습자료 복붙
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [15]:
import json

# 클로드 개별 업무 지시용 함수
def run_prompt(test_case):
    """프롬프트와 테스트 케이스 입력값을 병합한 후 결과를 반환"""
    prompt = f"""
    다음 작업을 수행하라:
    
    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation

    {test_case["task"]}
    """

    
    messages = []
    add_user_message(messages, prompt)
    # 맥스 토큰 내로 점수를 계산하기 위한 간결화 system prompt
    # python json regex 명시하지 않고도 셋 중 하나의 형식으로 시작하게 만드는 message refilling
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

# 클로드 개별 항목 검토 및 채점용 함수
def run_test_case(test_case):
    """run_prompt를 호출한 후 결과를 채점"""
    # 문제 먼저 풀고
    output = run_prompt(test_case)

    # 채점로직_ 그 답을 검토시키고
    model_grade= grade_by_model(test_case, output)
    # 검토 결과에서 점수만 추출
    model_score = model_grade["score"]
    # 검토 결과에서 설명만 추출
    reasoning = model_grade["reasoning"]

    # 코드 기반 채점에 관한 score 추가 후 모델 기반 채점 값과 평균(가중치를 동일하게 설정)하여 점수 산정
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

# 파이썬에서 평균 계산기 호출

from statistics import mean

# 위 함수를 활용하여 수행한 작업 및 이에 대한 채점 결과를 순차적으로 제시
def run_eval(dataset):
    """데이터셋을 로드하고 각 테스트 케이스에 run_test_case를 호출"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # 모든 테스트 케이스의 점수만 뽑아내어 이를 평균하여 제시
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

with open("dataset3.json", "r") as f:
    dataset = json.load(f)


In [16]:
results = run_eval(dataset)
# 결과값을 unicode가 아닌 한글로 인출하기 위해 ensure_ascii=False를 추가
json.dumps(results, indent=2, ensure_ascii=False)

Average score: 7.666666666666667


'[\n  {\n    "output": "\\nimport re\\nimport json\\n\\ndef parse_cloudwatch_log(log_entry):\\n    pattern = r\'^(\\\\d{4}-\\\\d{2}-\\\\d{2}T\\\\d{2}:\\\\d{2}:\\\\d{2}\\\\.\\\\d{3}Z)\\\\s+\\\\[(\\\\w+)\\\\]\\\\s+(.+)$\'\\n    match = re.match(pattern, log_entry)\\n    \\n    if match:\\n        return {\\n            \\"timestamp\\": match.group(1),\\n            \\"log_level\\": match.group(2),\\n            \\"message\\": match.group(3)\\n        }\\n    return None\\n\\nlog_example = \\"2024-01-15T10:30:45.123Z [INFO] Application started successfully\\"\\nresult = parse_cloudwatch_log(log_example)\\nprint(json.dumps(result, indent=2))\\n",\n    "test_case": {\n      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message using a regular expression",\n      "format": "regex",\n      "solution_criteria": "The regex must correctly capture three groups: ISO 8601 timestamp, log level (INFO, ERROR, WARNING, DEBUG), and the remaining message text from a